In [11]:
import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import AdamW
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
import time
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')
tf.config.optimizer.set_jit(True)

In [12]:
data = pd.read_csv('data/train.csv')

X = data.drop('label', axis=1).values.astype('float32')
y = data['label'].values

scaler = MinMaxScaler()
n_X = scaler.fit_transform(X)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

learning_rate = [1e-3]
batch_params = [256]
epoch_params = [50,100]

best_model = [None, None]
best_auc_roc = 0
for lr in learning_rate:
    for batch in batch_params:
        for epoch in epoch_params:
            auc_scores = []
            start_time = time.time()
            for train_idx, val_idx in skf.split(n_X, y):
                X_train, X_val = n_X[train_idx], n_X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]

                model = Sequential([
                Input(shape=(n_X.shape[1],)),

                Dense(64, activation='relu'),
                BatchNormalization(),
                Dropout(0.2),

                Dense(128, activation='relu'),
                BatchNormalization(),
                Dropout(0.3),

                Dense(256, activation='relu'),
                BatchNormalization(),
                Dropout(0.3),

                Dense(1, activation='sigmoid')
                ])


                model.compile(
                    loss='binary_crossentropy',
                    optimizer=AdamW(learning_rate=lr),
                    metrics=[tf.keras.metrics.AUC(name="auc")]
                )

                model.fit(X_train, y_train, epochs=epoch, batch_size=batch, verbose=0)
                score = model.evaluate(X_val, y_val, verbose=0)

                auc_scores.append(score[1])  # auc

            avg_auc = sum(auc_scores) / len(auc_scores)
            print(f"Batch: {batch}, Epoch: {epoch}, Score: {avg_auc}, Time Taken:{time.time() - start_time}, learning rate: {lr}")

            if avg_auc > best_auc_roc:
                best_auc_roc = avg_auc
                best_model = [batch, epoch]

print(f"Best AUC score: {best_auc_roc}, from batch size {best_model[0]} and {best_model[1]} epochs.")

Batch: 256, Epoch: 50, Score: 0.7711671233177185, Time Taken:338.35283303260803, learning rate: 0.001
Batch: 256, Epoch: 100, Score: 0.7844124436378479, Time Taken:663.4562323093414, learning rate: 0.001
Best AUC score: 0.7844124436378479, from batch size 256 and 100 epochs.


In [ ]:
import numpy as np
data = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

X = data.drop('label', axis=1).values.astype('float32')
y = data['label'].values

scaler = MinMaxScaler()
n_X = scaler.fit_transform(X)
model = Sequential([
                    Input(shape=(n_X.shape[1],)),
                     Dense(64, activation='relu'),
                BatchNormalization(),
                Dropout(0.2),

                Dense(128, activation='relu'),
                BatchNormalization(),
                Dropout(0.3),

                Dense(256, activation='relu'),
                BatchNormalization(),
                Dropout(0.3),

                Dense(1, activation='sigmoid')
                ])

model.compile(
                    loss='binary_crossentropy',
                    optimizer=AdamW(learning_rate=.005),
                    metrics=[tf.keras.metrics.AUC(name="auc")]
                )

model.fit(X, y, epochs=50, batch_size=256, verbose=1)
preds = model.predict(test).flatten()

submission = pd.DataFrame({
     'Id': [f"{i:.18e}" for i in range(len(test))],
    "Predicted": preds
})

submission.to_csv("submission.csv", index=False)

Epoch 1/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - auc: 0.6128 - loss: 0.6907
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.6946 - loss: 0.6319
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7205 - loss: 0.6144
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - auc: 0.7297 - loss: 0.6065
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7369 - loss: 0.6006
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7434 - loss: 0.5958
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7459 - loss: 0.5931
Epoch 8/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7512 - loss: 0.5884
Epoch 9/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7522 - loss: 0.5871
Epoch 10/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7567 - loss: 0.5833
Epoch 11/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7578 - loss: 0.5823
Epoch 12/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - auc: 0.7597 - loss: 0.5801
Epoch 13/50
196/196 ━━━━━

In [ ]:
Batch: 128, Epoch: 50, Score: 0.7512327432632446
Batch: 128, Epoch: 100, Score: 0.7435911695162455
Batch: 256, Epoch: 50, Score: 0.7472689350446066
Batch: 256, Epoch: 100, Score: 0.7474610010782877
Batch: 512, Epoch: 50, Score: 0.7427608768145243
Batch: 512, Epoch: 100, Score: 0.747559388478597
Best AUC score: 0.7512327432632446, from batch size 128 and 50 epochs.
2)
Batch: 128, Epoch: 50, Score: 0.7672291219234466
Batch: 256, Epoch: 50, Score: 0.7587428271770478
Batch: 512, Epoch: 50, Score: 0.7540872097015381
Best AUC score: 0.7672291219234466, from batch size 128 and 50 epochs.